# 168 — Attention-Value Composition

How do attention patterns and value vectors combine? Pattern-value alignment,
source decomposition, value mixing, cross-layer composition, and
logit decomposition by source.

## Why This Matters

Composition attention value measures how components interact across layers to implement complex computations. Understanding composition — how one head's output feeds into another's input — is essential for tracing multi-step algorithms in transformer circuits.

**Key references:**
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis
- [Olsson et al. (2022) "In-context Learning and Induction Heads"](https://arxiv.org/abs/2209.11895) — Discovers induction heads and training phase transitions

In [ ]:
import jax
import jax.numpy as jnp
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.attention_value_composition import (
    pattern_value_alignment,
    source_value_decomposition,
    value_mixing_analysis,
    composition_with_previous_layer,
    attention_output_logit_decomposition,
)

In [ ]:
cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.arange(15)

In [ ]:
result = pattern_value_alignment(model, tokens, layer=0)
for h in result['per_head']:
    print(f"Head {h['head']}: corr={h['pattern_value_correlation']:+.4f}  "
          f"eff_contrib={h['effective_contribution']:.4f}  "
          f"max_attn={h['max_attention']:.3f}")

In [ ]:
result = source_value_decomposition(model, tokens, layer=0, top_k=5)
for h in result['per_head']:
    sources = ', '.join(f"pos{s['source']}({s['attention_weight']:.2f})" for s in h['top_sources'])
    print(f"Head {h['head']}: total={h['total_output_norm']:.4f}  "
          f"top_frac={h['top_source_fraction']:.3f}  [{sources}]")

In [ ]:
result = value_mixing_analysis(model, tokens, layer=0)
for h in result['per_head']:
    print(f"Head {h['head']}: entropy={h['mixing_entropy']:.3f}  "
          f"n_sources={h['n_effective_sources']:.1f}  "
          f"diversity={h['value_diversity']:.3f}")

In [ ]:
for l in range(1, 4):
    result = composition_with_previous_layer(model, tokens, layer=l)
    for h in result['per_head']:
        print(f"L{l}H{h['head']}: V-comp={h['max_v_composition']:.4f}  "
              f"best_prev=H{h['best_prev_head']}")

In [ ]:
result = attention_output_logit_decomposition(model, tokens, layer=0, top_k=3)
print(f"Target token: {result['target_token']}\n")
for h in result['per_head']:
    sources = ', '.join(f"pos{s['source']}({s['logit_contribution']:+.3f})" for s in h['top_sources'])
    print(f"Head {h['head']}: total={h['total_logit_contribution']:+.4f}  [{sources}]")